# Fabric field-level lineage demo

Run this notebook **inside a Microsoft Fabric workspace**. Structure:

1. **Helper functions** — generic HTTP/polling/traversal utilities.
2. **Authentication** — get a Power BI / Fabric access token.
3. **Get fields of all** — scan the tenant and extract every upstream item (Lakehouse/Warehouse/KQL database/...), every semantic-model column/measure, and a lightweight report index.
4. **Parse M queries** — resolve which upstream item/schema/table each semantic-model table reads from.
5. **Join** — combine items, semantic-model fields and the M-query resolution into one lineage table.
6. **Look up** — find one upstream field and see what it feeds downstream ("if I change X, what breaks?").
7. **Report field lineage (optional)** — export PBIR report definitions and map report/page/visual field references to the upstream lineage.

Sections 1–6 cover **Lakehouse/Warehouse/KQL Database → semantic-model column/measure** lineage using only the Scanner API. Section 7 optionally extends this to report/page/visual references using Fabric report-definition calls. The lightweight report → semantic-model reference table (`reports_index`) remains available without those calls.

Every item, semantic model and report carries its **workspace**, straight from the Scanner API. A **folder** (within the workspace) is captured on a best-effort basis: the documented Scanner API schema has no folder info, so no extra calls are made for folders. The notebook only reads `folderId`/`folders` if returned; otherwise `folder` is `None`. Optional section 7 requires additional report access, not folder access.

**Prerequisites** — the Scanner APIs need *either*:
- a user with the tenant-wide **Fabric Administrator** role running this notebook interactively (delegated auth via `notebookutils.credentials.getToken`), **or**
- a **service principal** added to a dedicated Microsoft Entra security group that a Fabric admin has scoped into *Admin API settings* for read-only access — no tenant-wide admin role is needed at run time. See [Enable service principal authentication for admin APIs](https://learn.microsoft.com/en-us/fabric/admin/enable-service-principal-admin-apis). This is the recommended approach for anything beyond ad-hoc/interactive use.

Either way, a Fabric admin must still do this **one-time** tenant configuration (see [Set up metadata scanning](https://learn.microsoft.com/en-us/fabric/admin/metadata-scanning-setup)):
- Enable *"Enhance admin APIs responses with detailed metadata"*.
- Enable *"Enhance admin APIs responses with DAX and mashup expressions"* (needed for the M/DAX parsing this notebook does).
- If using a service principal: enable *"Allow service principals to use read-only Admin APIs"*, scoped to the security group containing the app.

> This is a demo. M-query parsing is regex-based and covers the common shapes (`Lakehouse.Contents`, `Sql.Database`, `AzureDataExplorer.Contents`, `Kusto.Contents`). It won't catch every hand-crafted mashup.


## 1. Helper functions

Generic utilities with no dependency on auth or scan state: HTTP calls, a long-running-operation poller, and two small traversal generators reused throughout the rest of the notebook.


In [ ]:
import re, time, json, requests
import pandas as pd
from collections import Counter


def _get(url, headers):
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    return r.json()


def _post(url, headers, body=None):
    r = requests.post(url, headers=headers, data=json.dumps(body) if body else None)
    r.raise_for_status()
    return r.json() if r.text else {}


def _poll_until(check_fn, succeeded="Succeeded", failed="Failed", delay=3,
                max_polls=None, label="operation"):
    """Call check_fn() repeatedly until it returns `succeeded`. Raises if it
    returns `failed` or after `max_polls` tries. Used by the scanner-API poll
    in step 3."""
    polls = 0
    while True:
        status = check_fn()
        if status == succeeded:
            return status
        if status == failed:
            raise RuntimeError(f"{label} failed (status={status!r})")
        polls += 1
        if max_polls is not None and polls >= max_polls:
            raise RuntimeError(f"{label} polling timed out")
        time.sleep(delay)


def iter_list_keys(ws):
    """Yield (key, items) for every non-empty list-valued key in a workspace dict."""
    for key, items in ws.items():
        if isinstance(items, list) and items:
            yield key, items


def iter_datasets(workspaces):
    """Yield (workspace, dataset) for every semantic model across all workspaces."""
    for w in workspaces:
        for ds in w.get("datasets", []) or []:
            yield w, ds


## 2. Authentication

`notebookutils.credentials.getToken` only accepts the named audiences `pbi`, `storage`, `keyvault` and `kusto` — there is no `fabric` audience (requesting one raises `"... is not a valid resource"`). The `pbi` token is valid for the Power BI Admin REST API below, and for Fabric REST APIs in general.

By default this authenticates as the **signed-in user** (works when running interactively in a Fabric workspace). Set `USE_SPN = True` below to instead authenticate as a **service principal** via MSAL client-credentials — useful for scheduled/unattended runs. The SPN needs:
- *"Allow service principals to use read-only Admin APIs"* enabled for its security group (see [Enable service principal authentication for admin APIs](https://learn.microsoft.com/en-us/fabric/admin/enable-service-principal-admin-apis)), and
- the `Tenant.Read.All` Power BI Service **application** permission, admin-consented.


In [ ]:
# Set to True to authenticate as a service principal instead of the signed-in
# user (e.g. for scheduled/unattended runs). See the prerequisites above.
USE_SPN = False
SPN_TENANT_ID = ""      # Microsoft Entra tenant ID
SPN_CLIENT_ID = ""      # App registration (client) ID
SPN_CLIENT_SECRET = ""  # App registration client secret

if USE_SPN:
    try:
        from msal import ConfidentialClientApplication
    except ImportError as e:
        raise ImportError('msal package not found -- run "%pip install msal" in a '
                           "cell above and retry.") from e

    # Prefer pulling the secret from Key Vault rather than hardcoding it above, e.g.:
    # SPN_CLIENT_SECRET = notebookutils.credentials.getSecret(
    #     "https://{vaultName}.vault.azure.net/", "{secretName}")
    app = ConfidentialClientApplication(
        SPN_CLIENT_ID,
        authority=f"https://login.microsoftonline.com/{SPN_TENANT_ID}",
        client_credential=SPN_CLIENT_SECRET,
    )
    result = app.acquire_token_for_client(scopes=["https://analysis.windows.net/powerbi/api/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"SPN token acquisition failed: {result.get('error_description')}")
    token = result["access_token"]
else:
    # notebookutils.credentials.getToken only accepts the named audiences
    # "pbi", "storage", "keyvault", "kusto" -- there is no "fabric" audience.
    token = notebookutils.credentials.getToken("pbi")

HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


## 3. Get fields of all

Run a tenant-wide scan with the [Fabric Scanner APIs](https://learn.microsoft.com/en-us/fabric/governance/metadata-scanning-overview), then extract raw fact tables from the result -- everything, including workspace/folder, comes only from this one scan (no extra per-workspace calls):
- `artifact_by_id` — every Lakehouse / Warehouse / KQL Database (Eventhouse) / etc. item in the tenant, with its workspace and (best-effort) folder.
- `model_fields` — every column and measure of every semantic model, with its workspace/folder, not yet linked to an upstream source.
- `reports_index` — a lightweight workspace/folder/semantic-model reference for every Power BI report (no field/visual-level detail).


In [ ]:
PBI_BASE = "https://api.powerbi.com/v1.0/myorg/admin"
BATCH_SIZE = 100             # workspaces per scan call (max 100)
MODIFIED_SINCE_DAYS = None   # how far back to look for modified workspaces; None = all

# 3a. Which workspaces to scan
url = f"{PBI_BASE}/workspaces/modified"
if MODIFIED_SINCE_DAYS is not None:
    from datetime import datetime, timedelta, timezone
    since = (datetime.now(timezone.utc) - timedelta(days=MODIFIED_SINCE_DAYS)).strftime("%Y-%m-%dT%H:%M:%S.0000000Z")
    url += f"?modifiedSince={since}"
ws_ids = [w["id"] for w in _get(url, HEADERS)]
print(f"Workspaces to scan: {len(ws_ids)}")

# 3b. Kick off scans in batches, poll until done, fetch results
scan_ids = []
post_url = (f"{PBI_BASE}/workspaces/getInfo"
            "?lineage=true&datasourceDetails=true&datasetSchema=true"
            "&datasetExpressions=true&getArtifactUsers=false")
for i in range(0, len(ws_ids), BATCH_SIZE):
    batch = ws_ids[i:i + BATCH_SIZE]
    resp = _post(post_url, HEADERS, {"workspaces": batch})
    scan_ids.append(resp["id"])
print(f"Scans started: {len(scan_ids)}")

workspaces = []
for sid in scan_ids:
    _poll_until(lambda sid=sid: _get(f"{PBI_BASE}/workspaces/scanStatus/{sid}", HEADERS)["status"],
                label=f"Scan {sid}")
    result = _get(f"{PBI_BASE}/workspaces/scanResult/{sid}", HEADERS)
    workspaces.extend(result.get("workspaces", []))
print(f"Workspaces returned: {len(workspaces)}")


In [ ]:
# 3c. Index every upstream item (Lakehouse / Warehouse / KQL database / ...)
_KEY_TO_KIND = {}
def _add_kind(kind, *aliases):
    for a in aliases:
        _KEY_TO_KIND[a.lower()] = kind

_add_kind("Lakehouse",            "lakehouse",  "lakehouses")
_add_kind("Warehouse",            "warehouse",  "warehouses")
_add_kind("KQLDatabase",          "kqldatabase", "kqldatabases")
_add_kind("KQLDataset",           "kqldataset", "kqldatasets")
_add_kind("SQLDatabase",          "sqldatabase", "sqldatabases")
_add_kind("SQLAnalyticsEndpoint", "sqlanalyticsendpoint", "sqlanalyticsendpoints")
_add_kind("MirroredDatabase",     "mirroreddatabase", "mirroreddatabases")
_add_kind("MirroredWarehouse",    "mirroredwarehouse", "mirroredwarehouses")
_add_kind("Datamart",             "datamart", "datamarts")
_add_kind("Eventhouse",           "eventhouse", "eventhouses")

artifact_by_id = {}          # artifact_id -> {"kind","name","workspace","workspaceId","folder"}
artifacts_by_workspace = {}  # workspace_id -> [artifact_id, ...]
datasource_by_id = {}        # datasource_instance_id -> connection details

# Folder support is best-effort only: the Scanner API's documented response
# schema (Admin - WorkspaceInfo GetScanResult) has no "folders" list or
# "folderId" field for any item type. We deliberately make no extra REST
# calls and assume no extra workspace access to get folder names -- we just
# opportunistically read folderId/folders if the scan happens to include
# them; otherwise "folder" stays None.
folder_by_id = {}  # folder_id -> {"name": ..., "parentFolderId": ...}
for w in workspaces:
    for fo in w.get("folders", []) or []:
        if isinstance(fo, dict) and fo.get("id"):
            folder_by_id[fo["id"]] = {"name": fo.get("displayName") or fo.get("name"),
                                       "parentFolderId": fo.get("parentFolderId")}


def _folder_path(folder_id):
    """Best-effort folder path (e.g. 'Sales/Archive'). Falls back to the raw
    folder ID if names can't be resolved, or None if there's no folder info."""
    if not folder_id:
        return None
    parts = []
    seen = set()
    fid = folder_id
    while fid and fid not in seen:
        seen.add(fid)
        info = folder_by_id.get(fid)
        if not info:
            break
        parts.append(info["name"] or fid)
        fid = info.get("parentFolderId")
    return "/".join(reversed(parts)) if parts else folder_id


def _register_item(wid, wname, aid, aname, kind, folder_id=None):
    if not aid:
        return
    artifact_by_id[aid] = {"kind": kind, "name": aname, "workspace": wname,
                            "workspaceId": wid, "folder": _folder_path(folder_id)}
    artifacts_by_workspace.setdefault(wid, []).append(aid)


for w in workspaces:
    wid, wname = w.get("id"), w.get("name")

    # Most item lists are plain top-level keys (e.g. "lakehouses": [...]).
    for key, items in iter_list_keys(w):
        kind = _KEY_TO_KIND.get(key.lower())
        if kind:
            for a in items:
                if isinstance(a, dict):
                    _register_item(wid, wname, a.get("id"), a.get("displayName") or a.get("name"),
                                    kind, a.get("folderId"))

    # Some tenants/API versions instead return a flat "artifacts" list with an
    # explicit artifactType field.
    for a in w.get("artifacts", []) or []:
        if not isinstance(a, dict):
            continue
        kind = _KEY_TO_KIND.get((a.get("artifactType") or a.get("type") or "").lower())
        if kind:
            _register_item(wid, wname, a.get("id"), a.get("displayName") or a.get("name"),
                            kind, a.get("folderId"))

    for di in w.get("datasourceInstances", []) or []:
        did = di.get("datasourceId") or di.get("id")
        if did:
            datasource_by_id[did] = di

print(f"Upstream items indexed: {len(artifact_by_id)}")
for k, v in sorted(Counter(info["kind"] for info in artifact_by_id.values()).items()):
    print(f"  {k}: {v}")
with_folder = sum(1 for info in artifact_by_id.values() if info["folder"])
print(f"Items with folder info: {with_folder}/{len(artifact_by_id)} "
      f"(best-effort -- the Scanner API schema doesn't guarantee it)")


In [ ]:
# 3d. Every column/measure of every semantic model, before we know its upstream source
model_fields = []
for w, ds in iter_datasets(workspaces):
    ds_folder = _folder_path(ds.get("folderId"))
    for t in ds.get("tables", []) or []:
        for c in t.get("columns", []) or []:
            model_fields.append({
                "workspace_id": w.get("id"), "workspace": w.get("name"), "folder": ds_folder,
                "dataset_id": ds.get("id"), "dataset": ds.get("name"),
                "table": t.get("name"), "field": c.get("name"), "field_type": "Column",
            })
        for m in t.get("measures", []) or []:
            model_fields.append({
                "workspace_id": w.get("id"), "workspace": w.get("name"), "folder": ds_folder,
                "dataset_id": ds.get("id"), "dataset": ds.get("name"),
                "table": t.get("name"), "field": m.get("name"), "field_type": "Measure",
                "measure_expression": m.get("expression") or "",
            })

print(f"Semantic-model fields indexed: {len(model_fields)}")

# 3e. Lightweight report index (workspace/folder/semantic model), sourced
# entirely from the scan result above -- no extra REST calls or workspace
# access. Optional section 7 separately inspects report definitions;
# this is just a reference table of where reports live.
dataset_by_id = {ds["id"]: {"name": ds.get("name"), "workspace": w.get("name")}
                 for w, ds in iter_datasets(workspaces) if ds.get("id")}

reports_index = []
for w in workspaces:
    for rp in w.get("reports", []) or []:
        ds_info = dataset_by_id.get(rp.get("datasetId")) if rp.get("datasetId") else None
        reports_index.append({
            "workspace": w.get("name"),
            "folder": _folder_path(rp.get("folderId")),
            "report": rp.get("name"),
            "semantic_model": ds_info["name"] if ds_info else None,
            "semantic_model_workspace": ds_info["workspace"] if ds_info else None,
        })

reports_df = pd.DataFrame(reports_index)
print(f"Reports indexed: {len(reports_df)}")
reports_df.head(20)


## 4. Parse M queries to resolve each table's upstream item

For each semantic-model table, inspect its Power Query M source (and a few fallbacks, in priority order) to work out which upstream item/schema/table it reads from.


In [ ]:
RE_LAKEHOUSE = re.compile(r'Lakehouse\.Contents\([^)]*\)(?P<rest>.*?)(?=(?:in\s|\Z))', re.S | re.I)
RE_SQL       = re.compile(r'Sql\.Databases?\(\s*"(?P<srv>[^"]+)"[^)]*\)', re.I)
RE_KQL       = re.compile(r'(?:AzureDataExplorer|Kusto)\.Contents\(\s*"(?P<srv>[^"]+)"', re.I)
RE_NAV       = re.compile(r'\{\s*\[(?P<props>[^\]]+)\]\s*\}', re.S)
RE_PROP      = re.compile(r'(\w+)\s*=\s*"([^"]*)"')
RE_GUID      = re.compile(r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b")

_KIND_PRIORITY = {"Lakehouse": 0, "Warehouse": 1, "MirroredDatabase": 2,
                  "KQLDatabase": 3, "SQLAnalyticsEndpoint": 4, "SQLDatabase": 5, "Datamart": 6}


def _nav_props(expr):
    """Return list of dicts, one per {[...]} navigation step in M source."""
    return [dict(RE_PROP.findall(m.group("props"))) for m in RE_NAV.finditer(expr or "")]


def resolve_upstream(expr):
    """Return (kind, artifact_name, workspace_name, folder, schema, table)
    parsed from an M expression, or None. `folder` is only populated when the
    artifact could be matched back to artifact_by_id (best-effort, see step 3)."""
    if not expr:
        return None
    merged = {}
    for p in _nav_props(expr):
        merged.update(p)

    schema = merged.get("Schema")
    table  = merged.get("Item") or merged.get("Name")
    kind   = merged.get("ItemKind")
    art_id = merged.get("Id") or merged.get("ItemId")
    art_nm = merged.get("LakehouseName") or merged.get("WarehouseName") or merged.get("DatabaseName")

    artifact, workspace, folder = None, None, None
    if art_id and art_id in artifact_by_id:
        info = artifact_by_id[art_id]
        artifact, workspace, kind, folder = info["name"], info["workspace"], info["kind"], info["folder"]
    elif art_nm:
        artifact = art_nm

    if not kind:
        if RE_LAKEHOUSE.search(expr): kind = "Lakehouse"
        elif RE_KQL.search(expr):     kind = "KQLDatabase"
        elif RE_SQL.search(expr):     kind = "Warehouse/SQL"

    if not (kind or artifact or table):
        return None
    return (kind or "Unknown", artifact or "?", workspace or "?", folder, schema or "?", table or "?")


def _collect_expression(table_obj):
    """Return the concatenated M expression for a semantic-model table."""
    chunks = []
    src = table_obj.get("source")
    if isinstance(src, list):
        for s in src:
            if isinstance(s, dict) and s.get("expression"):
                chunks.append(s["expression"])
            elif isinstance(s, str):
                chunks.append(s)
    elif isinstance(src, dict) and src.get("expression"):
        chunks.append(src["expression"])
    elif isinstance(src, str):
        chunks.append(src)

    for p in table_obj.get("partitions", []) or []:
        psrc = (p or {}).get("source") or {}
        if isinstance(psrc, dict) and psrc.get("expression"):
            chunks.append(psrc["expression"])
        elif isinstance(psrc, str):
            chunks.append(psrc)

    if table_obj.get("expression"):
        chunks.append(table_obj["expression"])

    return "\n".join(chunks)


def _table_entity_schema(table_obj):
    """Return (entity_name, schema_name) from a Direct Lake table.source, if present."""
    entity, schema = None, None
    src = table_obj.get("source")
    if isinstance(src, list):
        for s in src:
            if isinstance(s, dict):
                entity = entity or s.get("entityName") or s.get("Item")
                schema = schema or s.get("schemaName") or s.get("Schema")
    elif isinstance(src, dict):
        entity = src.get("entityName") or src.get("Item")
        schema = src.get("schemaName") or src.get("Schema")
    return entity, schema


def _dataset_upstream_ids(ds):
    """Return artifact IDs the dataset lists as upstream (direct link)."""
    ids = []
    for key in ("upstreamArtifacts", "upstreamDatasources", "upstreamDatasets", "upstreamDataflows"):
        for u in ds.get(key, []) or []:
            if isinstance(u, dict):
                aid = u.get("artifactId") or u.get("id") or u.get("targetId")
                if aid:
                    ids.append(aid)
            elif isinstance(u, str):
                ids.append(u)
    return ids


def _dataset_datasource_ids(ds):
    """Return datasourceInstance IDs the dataset uses."""
    ids = []
    for u in ds.get("datasourceUsages", []) or []:
        if isinstance(u, dict):
            did = u.get("datasourceInstanceId") or u.get("id")
            if did:
                ids.append(did)
    return ids


def _guids_in_dataset(ds):
    """Recursively yield every GUID that appears anywhere in the dataset dict."""
    stack = [ds]
    seen = set()
    while stack:
        v = stack.pop()
        if isinstance(v, dict):
            stack.extend(v.values())
        elif isinstance(v, list):
            stack.extend(v)
        elif isinstance(v, str):
            for g in RE_GUID.findall(v):
                if g not in seen:
                    seen.add(g)
                    yield g


def _pick_same_workspace(ids):
    """From a list of artifact IDs, pick the highest-priority upstream item."""
    infos = [artifact_by_id[a] for a in ids if a in artifact_by_id]
    if not infos:
        return None
    infos.sort(key=lambda i: _KIND_PRIORITY.get(i["kind"], 99))
    return infos[0]


def resolve_table_upstream(table_obj, ds, ds_workspace_id):
    """Try each resolver in turn. Returns (kind, art, ws, folder, schema, up_tbl, how) or None."""
    entity, schema = _table_entity_schema(table_obj)
    tname = table_obj.get("name")

    up = resolve_upstream(_collect_expression(table_obj))
    if up:
        return (*up, "M-parse")

    for aid in _dataset_upstream_ids(ds):
        if aid in artifact_by_id:
            info = artifact_by_id[aid]
            return (info["kind"], info["name"], info["workspace"], info["folder"],
                    schema or "?", entity or tname or "?", "upstreamArtifacts")

    for did in _dataset_datasource_ids(ds):
        di = datasource_by_id.get(did) or {}
        details = di.get("connectionDetails") or {}
        srv = (details.get("server") or details.get("url") or "").lower()
        for aid, info in artifact_by_id.items():
            if info["name"] and info["name"].lower() in srv:
                return (info["kind"], info["name"], info["workspace"], info["folder"],
                        schema or "?", entity or tname or "?", "datasourceUsage")

    info = _pick_same_workspace([g for g in _guids_in_dataset(ds) if g in artifact_by_id])
    if info:
        return (info["kind"], info["name"], info["workspace"], info["folder"],
                schema or "?", entity or tname or "?", "guid-in-dataset")

    info = _pick_same_workspace(artifacts_by_workspace.get(ds_workspace_id, []))
    if info:
        return (info["kind"], info["name"], info["workspace"], info["folder"],
                schema or "?", entity or tname or "?", "same-workspace guess")

    return None


# Resolve every table once; (workspace_id, dataset_id, table_name) -> result or None
table_upstream = {}
for w, ds in iter_datasets(workspaces):
    wid = w.get("id")
    for t in ds.get("tables", []) or []:
        table_upstream[(wid, ds.get("id"), t.get("name"))] = resolve_table_upstream(t, ds, wid)

resolved = sum(1 for v in table_upstream.values() if v)
print(f"Tables resolved to an upstream item: {resolved}/{len(table_upstream)}")
if resolved < len(table_upstream):
    print("Unresolved tables are usually Direct Lake models whose Lakehouse/Warehouse either")
    print("lives in a workspace the token can't see, or wasn't included in this scan.")


## 5. Join items, semantic-model fields and M-query results into one lineage table

One row per upstream field → downstream semantic-model column/measure, each carrying its workspace and (best-effort) folder on both sides.


In [ ]:
RE_DAX_COLREF = re.compile(r"'?(?P<tbl>[A-Za-z0-9_ ]+?)'?\[(?P<col>[^\]]+)\]")

LINEAGE_COLUMNS = [
    "upstream", "upstream_kind", "upstream_item", "upstream_workspace", "upstream_folder",
    "upstream_field",
    "downstream_kind", "downstream_item", "downstream_workspace", "downstream_folder",
    "resolution",
    "model_workspace_id", "model_id", "model_table", "model_field", "model_field_type",
]


def _lineage_row(up, downstream_kind, downstream_item, downstream_workspace, downstream_folder, field, model_field):
    kind, art, ws, folder, schema, up_tbl, how = up
    return {
        "upstream":       f"{kind}:{art}.{schema}.{up_tbl}.{field}",
        "upstream_kind":  kind,
        "upstream_item":  art,
        "upstream_workspace": ws,
        "upstream_folder": folder,
        "upstream_field": f"{schema}.{up_tbl}.{field}",
        "downstream_kind":    downstream_kind,
        "downstream_item":    downstream_item,
        "downstream_workspace": downstream_workspace,
        "downstream_folder": downstream_folder,
        "resolution": how,
        "model_workspace_id": model_field["workspace_id"],
        "model_id": model_field["dataset_id"],
        "model_table": model_field["table"],
        "model_field": model_field["field"],
        "model_field_type": model_field["field_type"],
    }


# Column names per (workspace, dataset, table), needed to validate DAX measure
# references below (a measure can reference columns from any table in its model).
table_cols = {}
for f in model_fields:
    if f["field_type"] == "Column":
        table_cols.setdefault((f["workspace"], f["dataset"], f["table"]), set()).add(f["field"])

rows = []
for f in model_fields:
    ds_key = (f["workspace_id"], f["dataset_id"])

    if f["field_type"] == "Column":
        up = table_upstream.get((*ds_key, f["table"]))
        if up:
            rows.append(_lineage_row(up, "SemanticModel.Column",
                                      f"{f['dataset']}.{f['table']}.{f['field']}",
                                      f["workspace"], f["folder"], f["field"], f))
    else:  # Measure
        refs = {(rt.strip(), rc.strip()) for rt, rc in RE_DAX_COLREF.findall(f.get("measure_expression", ""))}
        for rt, rc in refs:
            up = table_upstream.get((*ds_key, rt))
            if not up or rc not in table_cols.get((f["workspace"], f["dataset"], rt), set()):
                continue
            rows.append(_lineage_row(up, "SemanticModel.Measure",
                                      f"{f['dataset']}.[{f['field']}]",
                                      f["workspace"], f["folder"], rc, f))

lineage = pd.DataFrame(rows, columns=LINEAGE_COLUMNS).drop_duplicates()
if not lineage.empty:
    lineage = lineage.sort_values(["upstream", "downstream_item"])

print(f"Lineage edges: {len(lineage)}")
if not lineage.empty:
    print("Edges by resolution method:")
    print(lineage["resolution"].value_counts().to_string())
else:
    print("No lineage edges found. Check the 'Tables resolved' count in step 4 above.")
lineage.head(20)


## 6. Look up a field's impact

`lookup(item=..., schema=..., table=..., fields=...)` filters `lineage` for the downstream semantic-model columns/measures fed by an upstream field — "if I change X, what breaks?" Each argument is an optional case-insensitive substring match; `fields` accepts a single name or a list. The most recent lookup also saves its full matching rows as `selected_lineage`, which section 7 reuses automatically. No need to specify the field again.


In [ ]:
def apply_filters(df, item=None, schema=None, table=None, fields=None):
    """Filter a lineage-shaped DataFrame by upstream_item / schema / table / field(s).
    All matches are case-insensitive substring matches. `fields` may be a
    string or a list of strings (OR-combined)."""
    out = df
    if item:
        out = out[out["upstream_item"].str.contains(item, case=False, na=False, regex=False)]

    def _part(n):
        # Re-split upstream_field (schema.table.field) against the *current*
        # `out`, so an empty intermediate result never leaves a stale/None
        # lookup for a later filter to index into.
        if out.empty:
            return pd.Series([], dtype=str)
        return out["upstream_field"].str.split(".", n=2, expand=True)[n].fillna("")

    if schema:
        out = out[_part(0).str.contains(schema, case=False, na=False, regex=False)]
    if table:
        out = out[_part(1).str.contains(table, case=False, na=False, regex=False)]
    if fields:
        fl = [fields] if isinstance(fields, str) else list(fields)
        col = _part(2)
        mask = pd.Series(False, index=out.index)
        for f in fl:
            mask = mask | col.str.contains(f, case=False, na=False, regex=False)
        out = out[mask]
    return out


selected_lineage = None


def lookup(item=None, schema=None, table=None, fields=None) -> pd.DataFrame:
    """Return downstream usages of upstream fields matching the given filters."""
    global selected_lineage
    hits = apply_filters(lineage, item=item, schema=schema, table=table, fields=fields)
    selected_lineage = hits.copy()
    if hits.empty:
        print(f"No downstream usage for item={item!r}, schema={schema!r}, "
              f"table={table!r}, fields={fields!r}.")
        return hits
    print(f"Matched {hits['upstream_field'].nunique()} upstream field(s), "
          f"used by {hits['downstream_item'].nunique()} downstream object(s).")
    return hits[["upstream", "downstream_kind", "downstream_item", "downstream_workspace"]]


# Change the call below to explore. Only the LAST expression in a cell is shown.
lookup(item="Kaggle", table="betfront", fields="COUNTRY")

# --- Alternative views (comment out the call above and uncomment ONE) ---
#
# lookup(item="Kaggle")                       # everything fed by the Kaggle Lakehouse
# apply_filters(lineage)                      # the whole raw lineage table
#
# (apply_filters(lineage)                     # grouped "where is each field used"
#     .groupby(["upstream_kind", "upstream_workspace", "upstream_item", "upstream_field"])
#     .agg(downstream_count=("downstream_item", "nunique"),
#          downstream_items=("downstream_item", lambda s: sorted(set(s))))
#     .reset_index()
#     .sort_values("downstream_count", ascending=False))


## 7. Optional: extend field lineage to Power BI reports

Set `INCLUDE_REPORT_LINEAGE = True` below to fetch [report definitions](https://learn.microsoft.com/en-us/rest/api/fabric/report/items/get-report-definition) in PBIR format. This is separate from the tenant scan: the caller needs **read and write permissions on each report**, with delegated scope `Report.ReadWrite.All` or `Item.ReadWrite.All` for user authentication. Tenant-wide Fabric Administrator/read-only Admin API access alone does **not** grant this access. Service principals also need appropriate workspace/report access and the tenant setting allowing service principals to call Fabric public APIs.

This step reuses `selected_lineage` from the last `lookup(...)` in section 6. Only reports connected to those semantic-model IDs are inspected, and only references to those selected fields appear in the final table. `REPORT_WORKSPACE_IDS` can further limit report workspaces; `None` imposes no extra workspace restriction. An empty selection makes no report API calls.

The final output is **one joined table: Lakehouse field → semantic-model column/measure → Power BI report**, retaining report/page/visual context. Set `SHOW_REPORT_ERRORS = True` to also display diagnostic details. Diagnostics are always retained in `report_scan_status`; a brief warning indicates incomplete coverage even when details are hidden. Requests handle HTTP 202 operations and bounded HTTP 429 retries. Protected sensitivity labels, legacy/non-PBIR reports, unsupported workspaces and access failures are not evidence of no usage.

- `report_field_usage`: table-qualified column/measure references found in report, page and visual JSON, with report/model IDs and definition-part path. Includes filters as well as visual bindings; a reference is not proof that it is visible in the rendered report.
- `report_lineage`: the displayed joined table, restricted to `selected_lineage` and matched by model ID, table, field and field type (not display names). Reports without a matching selected field are excluded.
- `full_lineage()`: returns that same selected joined table; optional `item`, `schema`, `table`, `fields` arguments narrow it further. To select a different source field, rerun `lookup(...)` in section 6 and then rerun this section's final cell. Existing `lineage` and `lookup` remain model-only.

This parser supports PBIR `Column`/`Measure` expressions with `SourceRef.Entity` or query-local `SourceRef.Source` aliases. References whose source cannot be resolved are counted in the status table. Other expression types, measure-to-measure dependencies, dynamic field parameters and other indirect dependencies are not expanded; report edges inherit the upstream resolver's limitations. Always inspect coverage before concluding that a field is unused.


In [ ]:
import base64
from pathlib import PurePosixPath
from uuid import UUID

FABRIC_BASE = "https://api.fabric.microsoft.com/v1"
REPORT_USAGE_COLUMNS = [
    "report_workspace_id", "report_workspace", "report_folder", "report_id", "report",
    "model_id", "model_table", "model_field", "model_field_type", "page", "visual", "part",
]
REPORT_JOIN_COLUMNS = ["model_id", "model_table", "model_field", "model_field_type"]


def _report_retry_delay(response):
    delay = int(response.headers.get("Retry-After", "3"))
    if not 0 <= delay <= 300:
        raise ValueError(f"Retry-After {delay}s is outside the supported 0–300s wait; retry later")
    return delay


def _report_request(method, url, headers):
    for attempt in range(4):
        response = requests.request(method, url, headers=headers, timeout=60)
        if response.status_code != 429 or attempt == 3:
            response.raise_for_status()
            return response
        delay = _report_retry_delay(response)
        time.sleep(delay)


def get_report_definition(workspace_id, report_id, headers):
    workspace_id, report_id = str(UUID(workspace_id)), str(UUID(report_id))
    url = f"{FABRIC_BASE}/workspaces/{workspace_id}/reports/{report_id}/getDefinition?format=PBIR"
    response = _report_request("POST", url, headers)
    if response.status_code == 202:
        operation_id = str(UUID(response.headers["x-ms-operation-id"]))
        operation_url = f"{FABRIC_BASE}/operations/{operation_id}"
        for _ in range(120):
            time.sleep(_report_retry_delay(response))
            response = _report_request("GET", operation_url, headers)
            operation = response.json()
            if operation.get("status") == "Succeeded":
                response = _report_request("GET", operation_url + "/result", headers)
                break
            if operation.get("status") in ("Failed", "Cancelled"):
                raise RuntimeError(f"Report definition operation failed: {operation.get('error', operation)}")
        else:
            raise TimeoutError("Report definition operation exceeded 120 polls")
    payload = response.json()
    definition = payload.get("definition")
    if not isinstance(definition, dict) or not isinstance(definition.get("parts"), list):
        raise ValueError("Report API response has no definition.parts list")
    return definition


def _report_references(node, aliases=None):
    if isinstance(node, dict):
        aliases = dict(aliases or {})
        for source in node.get("From", []) or []:
            if isinstance(source, dict) and source.get("Name") and source.get("Entity"):
                aliases[source["Name"]] = source["Entity"]
        for kind in ("Column", "Measure"):
            ref = node.get(kind)
            if isinstance(ref, dict) and ref.get("Property"):
                source = ref.get("Expression", {}).get("SourceRef", {})
                table = source.get("Entity") or aliases.get(source.get("Source"))
                yield table, ref["Property"], kind
        for value in node.values():
            yield from _report_references(value, aliases)
    elif isinstance(node, list):
        for value in node:
            yield from _report_references(value, aliases)


def parse_report_definition(definition):
    parts = definition["parts"]
    if not any(p.get("path", "").startswith("definition/") and
               p.get("path", "").endswith(".json") for p in parts):
        raise ValueError("No PBIR definition JSON parts; legacy report format is not supported")
    rows, unresolved = [], 0
    for part in parts:
        path = part.get("path", "")
        if not path.startswith("definition/") or not path.endswith(".json"):
            continue
        if part.get("payloadType") != "InlineBase64":
            raise ValueError(f"Unsupported payload type for {path}: {part.get('payloadType')}")
        document = json.loads(base64.b64decode(part["payload"], validate=True).decode("utf-8-sig"))
        path_parts = PurePosixPath(path).parts
        page = path_parts[path_parts.index("pages") + 1] if "pages" in path_parts else None
        visual = path_parts[path_parts.index("visuals") + 1] if "visuals" in path_parts else None
        for table, field, kind in _report_references(document):
            if not table:
                unresolved += 1
                continue
            rows.append({"model_table": table, "model_field": field, "model_field_type": kind,
                         "page": page, "visual": visual, "part": path})
    return rows, unresolved


def join_report_lineage(model_lineage, usage):
    left, right = model_lineage.copy(), usage.copy()
    keys = []
    for column in REPORT_JOIN_COLUMNS:
        key = "_join_" + column
        keys.append(key)
        left[key] = left[column].astype("string").str.casefold()
        right[key] = right[column].astype("string").str.casefold()
    right = right.drop(columns=REPORT_JOIN_COLUMNS)
    return (left.merge(right, on=keys, how="inner", validate="many_to_many")
            .drop(columns=keys).drop_duplicates())


def collect_report_lineage(scanned_workspaces, model_lineage, headers, workspace_ids=None, show_errors=False):
    usage_rows, status_rows = [], []
    selected = None if workspace_ids is None else {str(w).casefold() for w in workspace_ids}
    model_ids = set(model_lineage["model_id"].dropna().astype(str).str.casefold())
    for workspace in scanned_workspaces:
        if selected is not None and str(workspace["id"]).casefold() not in selected:
            continue
        for report in workspace.get("reports", []) or []:
            if str(report.get("datasetId", "")).casefold() not in model_ids:
                continue
            context = {"report_workspace_id": workspace["id"], "report_workspace": workspace.get("name"),
                       "report_folder": _folder_path(report.get("folderId")),
                       "report_id": report["id"], "report": report.get("name"),
                       "model_id": report.get("datasetId")}
            status = {**context, "status": None, "references": 0, "unresolved_expressions": 0,
                      "unlinked_references": 0, "error": None}
            try:
                if not context["model_id"]:
                    raise ValueError("Scanner report metadata has no datasetId; model mapping unavailable")
                definition = get_report_definition(workspace["id"], report["id"], headers)
                refs, unresolved = parse_report_definition(definition)
                report_usage = pd.DataFrame([{**context, **ref} for ref in refs],
                                            columns=REPORT_USAGE_COLUMNS).drop_duplicates()
                linked = join_report_lineage(model_lineage, report_usage)
                linked_keys = {tuple(str(row[c]).casefold() for c in REPORT_JOIN_COLUMNS)
                               for row in linked.to_dict("records")}
                unlinked = sum(tuple(str(row[c]).casefold() for c in REPORT_JOIN_COLUMNS) not in linked_keys
                               for row in report_usage.to_dict("records"))
                status.update(references=len(report_usage), unresolved_expressions=unresolved,
                              unlinked_references=unlinked,
                              status="partial" if unresolved else ("ok" if not linked.empty else
                                     ("no_matching_references" if refs else "no_references")))
                usage_rows.extend(report_usage.to_dict("records"))
            except (requests.RequestException, ValueError, KeyError, UnicodeError, RuntimeError, TimeoutError) as error:
                status.update(status="error", error=str(error))
                if isinstance(error, requests.HTTPError) and error.response is not None:
                    status["error"] += " | " + error.response.text[:1000]
                if show_errors:
                    print(f"Report {context['report']!r} could not be inspected: {status['error']}")
            status_rows.append(status)
    usage = pd.DataFrame(usage_rows, columns=REPORT_USAGE_COLUMNS).drop_duplicates()
    status_columns = [c for c in REPORT_USAGE_COLUMNS if c not in
                      ("model_table", "model_field", "model_field_type", "page", "visual", "part")]
    status_columns += ["status", "references", "unresolved_expressions", "unlinked_references", "error"]
    status = pd.DataFrame(status_rows, columns=status_columns)
    if not status.empty:
        errors = status["status"].eq("error").sum()
        unresolved = status["unresolved_expressions"].sum()
        if errors or unresolved:
            print(f"Report coverage incomplete: {errors} report(s) failed, {unresolved} unresolved "
                  "expression(s). Details: report_scan_status / SHOW_REPORT_ERRORS.")
    return usage, join_report_lineage(model_lineage, usage), status


In [ ]:
INCLUDE_REPORT_LINEAGE = True
REPORT_WORKSPACE_IDS = None  # Or ["workspace-guid", ...] to limit report exports
SHOW_REPORT_ERRORS = False

report_field_usage = pd.DataFrame(columns=REPORT_USAGE_COLUMNS)
report_lineage = join_report_lineage(lineage.iloc[:0], report_field_usage)
report_scan_status = pd.DataFrame()
if INCLUDE_REPORT_LINEAGE:
    if "selected_lineage" not in globals() or selected_lineage is None:
        raise RuntimeError("Run lookup(...) in section 6 to select source fields before section 7")
    report_field_usage, report_lineage, report_scan_status = collect_report_lineage(
        workspaces, selected_lineage, HEADERS, REPORT_WORKSPACE_IDS, show_errors=SHOW_REPORT_ERRORS)
    if selected_lineage.empty:
        print("The section 6 lookup matched no source fields; no reports were inspected.")
    elif report_scan_status.empty:
        print("No reports reference the selected semantic models in the selected report workspaces.")
    if SHOW_REPORT_ERRORS and not report_scan_status.empty:
        display(report_scan_status)
else:
    print("Report inspection disabled. Set INCLUDE_REPORT_LINEAGE = True and rerun this cell.")


def full_lineage(item=None, schema=None, table=None, fields=None):
    if not INCLUDE_REPORT_LINEAGE:
        raise RuntimeError("Enable INCLUDE_REPORT_LINEAGE and run section 7 first")
    return apply_filters(report_lineage, item=item, schema=schema, table=table, fields=fields)


# After enabling report inspection:
# full_lineage()  # Same selected Lakehouse -> semantic model -> report table
# report_field_usage  # Includes references without an upstream lineage match
report_lineage
